# Gradients by Study Area

We describe the pilot area (Lolland-Falster) as flat. We check this relative to the other five study areas in the dataset by comparing the maximum gradients of their links. Gradients are not used in the splendor index, so this only supports the choice of pilot area.

<br><br>

## 0. Setup

### 0.1. Imports and Config

In [1]:
import lzma
import pickle

import numpy as np
import pandas as pd

from splendor.paths import bikenwloops_network_path

AREAS = ["bornholm", "funen", "jutland", "lollandfalster", "longland", "zealand"]
PILOT = "lollandfalster"
LIMITS = (4, 6) # Maximum gradient limits in percent

### 0.2. Data

In [4]:
def load_links(area: str) -> pd.DataFrame:
    with lzma.open(bikenwloops_network_path(area, suffix="0"), "rb") as f:
        G = pickle.load(f)
    return pd.DataFrame({
        "area": area,
        "length_m": G.es["weight"],
        "max_gradient": np.array(G.es["max_slope"]),
    })

links = pd.concat([load_links(area) for area in AREAS], ignore_index=True)
print(links.groupby("area")["max_gradient"].describe().round(2))

                 count  mean   std   min   25%   50%   75%     max
area                                                              
bornholm          89.0  5.45  3.29  0.19  3.05  5.23  6.72   19.57
funen           1240.0  4.80  3.68  0.00  2.82  4.31  6.07   68.40
jutland         6502.0  4.67  4.08  0.00  2.43  4.05  6.04  135.54
lollandfalster   638.0  2.49  2.10  0.14  1.43  2.01  3.07   29.19
longland          88.0  5.04  2.04  0.81  3.54  4.95  6.32   12.12
zealand         2257.0  4.56  3.02  0.00  2.78  4.05  5.62   51.40


<br><br>

## 1. Gradient Distribution

In [5]:
def summarize(group: pd.DataFrame) -> pd.Series:
    summary = {
        "links": len(group),
        "km": group["length_m"].sum() / 1000,
        "median": group["max_gradient"].median(),
        "p90": group["max_gradient"].quantile(.9),
    }
    for limit in LIMITS:
        above = group["max_gradient"] > limit
        summary[f"links_above_{limit}"] = above.mean()
        summary[f"km_above_{limit}"] = group.loc[above, "length_m"].sum() / group["length_m"].sum()
    return pd.Series(summary)

summary = links.groupby("area").apply(summarize).sort_values("median")
print(summary.round(3))

                 links         km  median    p90  links_above_4  km_above_4  \
area                                                                          
lollandfalster   638.0   1439.353   2.008  4.432          0.130       0.156   
jutland         6502.0  18279.638   4.046  8.249          0.508       0.589   
zealand         2257.0   5232.960   4.051  7.538          0.508       0.620   
funen           1240.0   2572.452   4.311  7.901          0.546       0.652   
longland          88.0    226.481   4.953  7.536          0.670       0.721   
bornholm          89.0    375.732   5.235  9.240          0.618       0.752   

                links_above_6  km_above_6  
area                                       
lollandfalster          0.034       0.045  
jutland                 0.253       0.306  
zealand                 0.209       0.280  
funen                   0.260       0.322  
longland                0.284       0.329  
bornholm                0.348       0.469  


<br><br>

## 2. Conclusion

Lolland-Falster is clearly the flattest study area. Its links have a median maximum gradient of 2.0% against 4.0-5.2% in the other areas, and only 16% of its network length exceeds 4% gradient (59-75% elsewhere) and 4.5% exceeds 6% gradient (28-47% elsewhere).